# Import packages

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

# Import EBSD data
###### Import the GOS and GAKAM data from the three Excel files into a concatenated dataframe called "df." This dataframe is containing the GOS and GAKAM values, grain indexes, and file names.

In [ ]:
all_data = []

for excel_file in os.listdir(os.getcwd()):
    if excel_file.endswith(".xlsx"):
        sample_name = os.path.splitext(excel_file)[0]
        excel_path = os.path.join(os.getcwd(), excel_file)
        
        df = pd.read_excel(excel_path, header=None, skiprows=1, usecols="A,B")
        df.columns = ["GOS", "GAKAM"]
        df["id"] = df.index + 1
        df["file"] = sample_name
        print("Readed file: ", excel_file)
        
        all_data.append(df)

df = pd.concat(all_data, ignore_index=True)
print("### Dataframe created ### \n", df)


# Principal component analysis (PCA)

##### Log10 transformation of GOS and GAKAM data

In [ ]:
df["logGOS"] = np.log10(df["GOS"])
df["logGAKAM"] = np.log10(df["GAKAM"])

##### Create dataset matrix X and standard scaling

In [ ]:
X = df[["logGOS", "logGAKAM"]].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

##### PCA projection on first principal component PC1 and store result in the dataframe

In [ ]:
pca = PCA(n_components=1)
X_pca_scaled = pca.fit_transform(X_scaled)
df["PC1"] = X_pca_scaled

# Gaussian mixture model (GMM)

##### GMM fit on PC1 scaled dataset with two populations: recrystallized and deformed grains

In [ ]:
gmm = GaussianMixture(n_components=2)
gmm.fit(X_pca_scaled)

##### Extracting Gaussian parameters

In [ ]:
means = gmm.means_.flatten()
covariances = np.sqrt(gmm.covariances_).flatten()
weights = gmm.weights_

##### Sort Gaussian by their mean (for left and right plot positions)

In [ ]:
sorted_idx = np.argsort(means)
means = means[sorted_idx]
covariances = covariances[sorted_idx]
weights = weights[sorted_idx]

##### Calculate probability density function (PDF) of each Gaussian

In [ ]:
x_vals = np.linspace(min(X_pca_scaled), max(X_pca_scaled), 1000).reshape(-1, 1)

gauss1 = (
    weights[0] * np.exp(-0.5 * ((x_vals - means[0]) / covariances[0]) ** 2)
    / (covariances[0] * np.sqrt(2 * np.pi))
)
gauss2 = (
    weights[1] * np.exp(-0.5 * ((x_vals - means[1]) / covariances[1]) ** 2)
    / (covariances[1] * np.sqrt(2 * np.pi))
)

##### Convert PDF in grain number for plot

In [ ]:
bin_width = (max(X_pca_scaled) - min(X_pca_scaled)) / 100
Nb_grains = len(X_pca_scaled)
gauss1 *= Nb_grains * bin_width
gauss2 *= Nb_grains * bin_width
combined_gauss = gauss1 + gauss2

##### Find Gaussian intersection: it defines the population threshold
###### The intersection of the two Gaussians corresponds to one root of a second-degree polynomial. The correct root lies between the means of the two Gaussians.

##### Calculate the second-degree polynomial parameters a, b and c

In [ ]:
a = 1 / covariances[0]**2 - 1 / covariances[1]**2
b = 2 * (means[1] / covariances[1]**2 - means[0] / covariances[0]**2)
c = (means[0]**2 / covariances[0]**2 - means[1]**2 / covariances[1]**2) - 2 * np.log((weights[0] * covariances[1]) / (weights[1] * covariances[0]))

##### Find the valid root correspond to the population threshold

In [ ]:
roots = np.roots([a, b, c])

valid_roots = roots[(roots > min(means[0], means[1])) & (roots < max(means[0], means[1]))]

if valid_roots.size > 0:
    intersection = valid_roots[0]
    print("### Population threshold found ###")
else:
    intersection = None

# Uncertainty bar: interquartile range (IQR)

##### Calculate IQR and the limits around the population threshold

In [ ]:
q1_g1 = np.percentile(X_pca_scaled[X_pca_scaled <= means[0]], 25)
q3_g1 = np.percentile(X_pca_scaled[X_pca_scaled <= means[0]], 75)
iqr_g1 = q3_g1 - q1_g1

q1_g2 = np.percentile(X_pca_scaled[X_pca_scaled >= means[1]], 25)
q3_g2 = np.percentile(X_pca_scaled[X_pca_scaled >= means[1]], 75)
iqr_g2 = q3_g2 - q1_g2

lower_bound_iqr = intersection - iqr_g1
upper_bound_iqr = intersection + iqr_g2

##### Conclude on grain class

In [ ]:
def determine_class(PC1):
    if PC1 <= lower_bound_iqr:
        return "ReX"
    elif PC1 >= upper_bound_iqr:
        return "Deformed"
    return "Uncertain"

df["class"] = df["PC1"].apply(determine_class)
print("### Grains are classified ### \n", df["class"])

# Plot the one-dimensional distribution and PCA-GMM results

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(X_pca_scaled, bins=100, alpha=0.5, color="gray", label="PC1 Data")

ax.plot(x_vals, gauss1, label="Left Gaussian", color="blue", linestyle="--")
ax.plot(x_vals, gauss2, label="Right Gaussian", color="green", linestyle="--")
ax.plot(x_vals, combined_gauss, label="Gaussian mixture", color="black", linestyle="-", alpha=0.5)

ax.fill_between(x_vals.flatten(), 0, max(combined_gauss) * 1.1,
                where=(x_vals.flatten() >= lower_bound_iqr) & (x_vals.flatten() <= intersection),
                color="green", alpha=0.2, label="Interquartile range")

ax.fill_between(x_vals.flatten(), 0, max(combined_gauss) * 1.1,
                where=(x_vals.flatten() >= intersection) & (x_vals.flatten() <= upper_bound_iqr),
                color="green", alpha=0.2)

ax.axvline(x=intersection, color="black", linestyle="--", label="Population threshold")

ax.set_ylim(0, 360)
ax.set_xlabel("PC1 (scaled)")
ax.set_ylabel("Grain count")
ax.set_title("Distribution PC1 and GMM fit")
ax.legend()

plt.show()

print("### PROCESS DONE ###")